In [2]:
import pandas as pd
import ast
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import networkx as nx
from rapidfuzz import process, fuzz

c:\Users\winni\anaconda3\envs\tfenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("Data/recipes/RecipeNLG.csv")

In [3]:
print(df.head())
print(df.shape)

   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   
3           3          Chicken Funny   
4           4   Reeses Cups(Candy)     

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small jar chipped beef, cut up", "4 boned ...   
2  ["2 (16 oz.) pkg. frozen corn", "1 (8 oz.) pkg...   
3  ["1 large whole chicken", "2 (10 1/2 oz.) cans...   
4  ["1 c. peanut butter", "3/4 c. graham cracker ...   

                                          directions  \
0  ["In a heavy 2-quart saucepan, mix brown sugar...   
1  ["Place chipped beef on bottom of baking dish....   
2  ["In a slow cooker, combine all ingredients. C...   
3  ["Boil and debone chicken.", "Put bite size pi...   
4  ["Combine first four ingredients and press in ...   

                                              link    source  \
0   www.cookbooks.com

In [4]:
df = df.drop_duplicates(
    subset=["title", "ingredients", "directions"]
)
print(df.shape)

(2231142, 7)


In [5]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower().strip()
    return text

In [6]:
df["title_clean"] = df["title"].apply(normalize_text)
df["directions_clean"] = df["directions"].apply(normalize_text)
df["ingredients"] = df["ingredients"].apply(ast.literal_eval)

In [7]:
UNITS = r"\b(c|cup|cups|tbsp|tablespoon|tablespoons|tsp|teaspoon|teaspoons|oz|ounce|ounces|lb|lbs|pound|pounds|g|gram|grams|kg|ml|l|pkg|package|packages|can|cans|jar|jars)\b"

def clean_ingredient(item):
    if pd.isna(item):
        return ""

    item = str(item).lower()

    # remove parenthetical package sizes like "(16 oz.)"
    item = re.sub(r"\([^)]*\)", " ", item)

    # remove fractions/numbers
    item = re.sub(r"\d+\s*/\s*\d+|\d+\.\d+|\d+", " ", item)

    # remove units
    item = re.sub(UNITS, " ", item)

    # remove punctuation
    item = re.sub(r"[^a-z\s]", " ", item)

    # normalize spaces
    item = " ".join(item.split())

    return item

In [8]:
df["ingredients_clean"] = df["ingredients"].apply(
    lambda ingredients: [clean_ingredient(x) for x in ingredients]
)

In [9]:
df["semantic_text"] = (
    df["title_clean"] + " | " +
    df["ingredients_clean"].apply(lambda x: " ".join(sorted(x)))
)

In [10]:
df[["title", "ingredients", "ingredients_clean", "semantic_text"]].head()

,title,ingredients,ingredients_clean,semantic_text
0,No-Bake Nut Cookies,"[1 c. firmly packed brown sugar, 1/2 c. evapor...","[firmly packed brown sugar, evaporated milk, v...",no-bake nut cookies | bite size shredded rice ...
1,Jewell Ball'S Chicken,"[1 small jar chipped beef, cut up, 4 boned chi...","[small chipped beef cut up, boned chicken brea...",jewell ball's chicken | boned chicken breasts ...
2,Creamy Corn,"[2 (16 oz.) pkg. frozen corn, 1 (8 oz.) pkg. c...","[frozen corn, cream cheese cubed, butter cubed...",creamy corn | butter cubed cream cheese cubed ...
3,Chicken Funny,"[1 large whole chicken, 2 (10 1/2 oz.) cans ch...","[large whole chicken, chicken gravy, cream of ...",chicken funny | box stove top stuffing chicken...
4,Reeses Cups(Candy),"[1 c. peanut butter, 3/4 c. graham cracker cru...","[peanut butter, graham cracker crumbs, melted ...",reeses cups(candy) | graham cracker crumbs lar...


In [11]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df["semantic_text"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

c:\Users\winni\anaconda3\envs\tfenv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\winni\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8221.93it/s]
BertModel LOAD REPORT fro

In [13]:
np.save("recipe_embeddings.npy", embeddings)

In [14]:
embeddings = np.load("recipe_embeddings.npy")

In [15]:
df.to_parquet("recipes_cleaned.parquet")

In [2]:
df = pd.read_parquet("recipes_cleaned.parquet")

In [17]:
embeddings = embeddings.astype(np.float16)

In [20]:
np.save("recipe_embeddings_fp16.npy", embeddings)

In [21]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

In [22]:
faiss.write_index(index, "recipe_index.faiss")

In [23]:
index = faiss.read_index("recipe_index.faiss")

In [24]:
embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

scores, neighbors = index.search(embeddings, k=10)

embeddings = np.load("recipe_embeddings_fp16.npy").astype("float32")

In [25]:
index.add(embeddings)

In [26]:
faiss.write_index(index, "recipe_index.faiss")

In [27]:
index = faiss.read_index("recipe_index.faiss")

In [28]:
scores, neighbors = index.search(embeddings, k=10)

In [29]:
np.save("recipe_scores.npy", scores)
np.save("recipe_neighbors.npy", neighbors)

In [30]:
scores = np.load("recipe_scores.npy")
neighbors = np.load("recipe_neighbors.npy")

load all

In [32]:
df = pd.read_parquet("recipes_cleaned.parquet")
embeddings = np.load("recipe_embeddings_fp16.npy").astype("float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, "recipe_index.faiss")

run similarity search

In [33]:
k = 10

scores, neighbors = index.search(embeddings, k)

np.save("recipe_scores.npy", scores)
np.save("recipe_neighbors.npy", neighbors)

create duplicate pairs

In [34]:
pairs = []
threshold = 0.95

for i in range(len(df)):
    for rank in range(1, k):  # skip itself at rank 0
        j = neighbors[i][rank]
        score = scores[i][rank]

        if score >= threshold:
            pairs.append((i, j, float(score)))

duplicate_pairs = pd.DataFrame(
    pairs,
    columns=["recipe_id", "match_id", "similarity"]
)

duplicate_pairs.to_parquet("duplicate_pairs.parquet")
duplicate_pairs.head()

,recipe_id,match_id,similarity
0,2,1524587,0.985050
1,4,685461,0.959535
2,4,605167,0.954776
3,4,412237,0.953966
4,4,820397,0.952771


inspect examples

In [35]:
sample = duplicate_pairs.sample(20, random_state=42)

for _, row in sample.iterrows():
    i = int(row["recipe_id"])
    j = int(row["match_id"])

    print("=" * 80)
    print("Similarity:", row["similarity"])
    print("A:", df.iloc[i]["title"])
    print(df.iloc[i]["ingredients_clean"])
    print()
    print("B:", df.iloc[j]["title"])
    print(df.iloc[j]["ingredients_clean"])

Similarity: 0.9561333656311035
A: Pecan Pie
['eggs' 'sugar' 'corn syrup' 'melted butter or margarine' 'pecans chopped'
 'pie crust shell']

B: Pecan Pie
['sugar' 'corn syrup' 'melted butter' 'eggs beaten' 'pecans' 'pie shell']
Similarity: 0.9866492748260498
A: Self-Filled Cupcakes
['two layer chocolate cake mix' 'cream cheese softened' 'sugar' 'egg'
 'dash of salt' 'chocolate chips']

B: Self-Filled Cupcakes
['chocolate cake mix' 'cream cheese softened' 'sugar' 'egg' 'dash of salt'
 'semi sweet chocolate chips']
Similarity: 0.9944033026695251
A: Lemon Poppy-Seed Bread
['lemon cake mix' 'lemon instant pudding mix' 'water' 'salad oil' 'eggs'
 'poppy seed']

B: Lemon Poppy Seed Bread
['lemon cake mix' 'instant lemon pudding mix' 'water' 'salad oil'
 'poppy seed' 'eggs']
Similarity: 0.9808543920516968
A: Double Layer Pumpkin Pie
['cream cheese softened' 'sugar' 'graham cracker crust'
 'jell o vanilla instant pudding' 'ground ginger' 'milk'
 'thawed cool whip' 'cold milk' 'pumpkin' 'ground 

remove mirrors

In [38]:
duplicate_pairs["recipe_min"] = duplicate_pairs[["recipe_id", "match_id"]].min(axis=1)
duplicate_pairs["recipe_max"] = duplicate_pairs[["recipe_id", "match_id"]].max(axis=1)

duplicate_pairs_unique = duplicate_pairs.drop_duplicates(
    subset=["recipe_min", "recipe_max"]
).copy()

duplicate_pairs_unique = duplicate_pairs_unique[
    ["recipe_min", "recipe_max", "similarity"]
].rename(columns={
    "recipe_min": "recipe_id",
    "recipe_max": "match_id"
})

duplicate_pairs_unique.to_parquet("duplicate_pairs_unique.parquet")

duplicate_pairs_unique.head()

,recipe_id,match_id,similarity
0,2,1524587,0.985050
1,4,685461,0.959535
2,4,605167,0.954776
3,4,412237,0.953966
4,4,820397,0.952771


reload state

In [3]:
# Load cleaned recipe dataset (can only be done after parquet creation, obviously)
df = pd.read_parquet("recipes_cleaned.parquet")

# Load duplicate candidate pairs
duplicate_pairs = pd.read_parquet("duplicate_pairs.parquet")

# Rebuild unique pair table
duplicate_pairs["recipe_min"] = duplicate_pairs[
    ["recipe_id", "match_id"]
].min(axis=1)

duplicate_pairs["recipe_max"] = duplicate_pairs[
    ["recipe_id", "match_id"]
].max(axis=1)

duplicate_pairs_unique = duplicate_pairs.drop_duplicates(
    subset=["recipe_min", "recipe_max"]
).copy()

duplicate_pairs_unique = duplicate_pairs_unique[
    ["recipe_min", "recipe_max", "similarity"]
].rename(columns={
    "recipe_min": "recipe_id",
    "recipe_max": "match_id"
})

print(df.shape)
print(duplicate_pairs_unique.shape)

(2231142, 11)
(1307986, 3)


new de-duplicate method

In [4]:
def ingredient_jaccard(a, b):
    set_a = set(a)
    set_b = set(b)

    if len(set_a) == 0 or len(set_b) == 0:
        return 0.0

    return len(set_a & set_b) / len(set_a | set_b)


def title_similarity(a, b):
    return fuzz.token_set_ratio(str(a), str(b)) / 100

In [5]:
hybrid_rows = []

for _, row in duplicate_pairs_unique.iterrows():
    i = int(row["recipe_id"])
    j = int(row["match_id"])

    semantic_score = float(row["similarity"])

    ingredient_score = ingredient_jaccard(
        df.iloc[i]["ingredients_clean"],
        df.iloc[j]["ingredients_clean"]
    )

    title_score = title_similarity(
        df.iloc[i]["title_clean"],
        df.iloc[j]["title_clean"]
    )

    hybrid_score = (
        0.50 * semantic_score +
        0.35 * ingredient_score +
        0.15 * title_score
    )

    hybrid_rows.append({
        "recipe_id": i,
        "match_id": j,
        "semantic_score": semantic_score,
        "ingredient_jaccard": ingredient_score,
        "title_similarity": title_score,
        "hybrid_score": hybrid_score
    })

hybrid_pairs = pd.DataFrame(hybrid_rows)

hybrid_pairs.to_parquet("hybrid_duplicate_pairs.parquet", index=False)

hybrid_pairs.head()

,recipe_id,match_id,semantic_score,ingredient_jaccard,title_similarity,hybrid_score
0,2,1524587,0.985050,0.714286,1.000000,0.892525
1,4,685461,0.959535,0.428571,0.590909,0.718404
2,4,605167,0.954776,0.428571,0.590909,0.716024
3,4,412237,0.953966,0.250000,0.758621,0.678276
4,4,820397,0.952771,0.250000,0.733333,0.673886


In [6]:
validated_pairs = hybrid_pairs[
    (hybrid_pairs["semantic_score"] >= 0.92) &
    (hybrid_pairs["ingredient_jaccard"] >= 0.50) &
    (hybrid_pairs["hybrid_score"] >= 0.75)
].copy()

validated_pairs.to_parquet("validated_duplicate_pairs_hybrid.parquet", index=False)

validated_pairs.head()

,recipe_id,match_id,semantic_score,ingredient_jaccard,title_similarity,hybrid_score
0,2,1524587,0.985050,0.714286,1.000000,0.892525
5,4,1662758,0.951752,0.500000,0.685714,0.753733
7,6,168952,0.972993,0.636364,1.000000,0.859224
9,6,87843,0.961833,0.545455,1.000000,0.821826
26,17,170615,0.993527,0.500000,1.000000,0.821764


create clusters

In [7]:
G = nx.Graph()

G.add_nodes_from(range(len(df)))

G.add_edges_from(
    validated_pairs[["recipe_id", "match_id"]].itertuples(index=False, name=None)
)

clusters = list(nx.connected_components(G))

clusters = [cluster for cluster in clusters if len(cluster) > 1]

len(clusters)

60726

create cluster tables

In [8]:
cluster_rows = []

for cluster_id, members in enumerate(clusters):
    for recipe_id in members:
        cluster_rows.append({
            "cluster_id": cluster_id,
            "recipe_id": recipe_id,
            "cluster_size": len(members)
        })

recipe_clusters_hybrid = pd.DataFrame(cluster_rows)

recipe_clusters_hybrid.to_parquet("recipe_clusters_hybrid.parquet", index=False)

recipe_clusters_hybrid.head()

,cluster_id,recipe_id,cluster_size
0,0,2,2
1,0,1524587,2
2,1,4,71
3,1,202117,71
4,1,87686,71


inspect clusters

In [9]:
recipe_clusters_hybrid["cluster_size"].describe()

recipe_clusters_hybrid["cluster_size"].value_counts().sort_index().head(20)

cluster_size
2     89422
3     23409
4     11488
5      6690
6      4836
7      3990
8      3080
9      2295
10     2150
11     1892
12     1440
13     1339
14     1260
15     1185
16      992
17     1003
18     1206
19     1083
20     1120
21      840
Name: count, dtype: int64

In [10]:
recipe_clusters_hybrid[["cluster_id", "cluster_size"]] \
    .drop_duplicates() \
    .sort_values("cluster_size", ascending=False) \
    .head(20)

,cluster_id,cluster_size
3750,25,6667
31555,103,4857
28286,100,3236
44151,156,3233
12196,43,2915
23521,72,2443
37062,113,2138
56620,237,2086
59735,252,1760
21723,69,1607


inspect one cluster

In [11]:
cluster_id = recipe_clusters_hybrid.sort_values(
    "cluster_size",
    ascending=False
)["cluster_id"].iloc[0]

ids = recipe_clusters_hybrid.loc[
    recipe_clusters_hybrid["cluster_id"] == cluster_id,
    "recipe_id"
].tolist()

df.loc[ids, ["title", "ingredients_clean", "link"]].head(30)

,title,ingredients_clean,link
425984,Oatmeal Cookies,"[butter, shortening, white sugar, brown sugar,...",www.cookbooks.com/Recipe-Details.aspx?id=897048
98304,Butter Cookies,"[butter, powdered sugar, flour, salt, vanilla]",www.cookbooks.com/Recipe-Details.aspx?id=354155
425990,Peanut Butter Cookies,"[crisco, brown sugar, beaten eggs, flour sifte...",www.cookbooks.com/Recipe-Details.aspx?id=729165
2195463,Simply Delicious Peanut Butter Cookies,"[creamy peanut butter, sugar, egg, vanilla ext...",www.food.com/recipe/simply-delicious-peanut-bu...
589833,Peanut Butter Cookies,"[shortening, butter, peanut butter, granulated...",www.cookbooks.com/Recipe-Details.aspx?id=197640
393233,Outrageous Chocolate Chip Cookies,"[granulated sugar, packed brown sugar, butter ...",www.cookbooks.com/Recipe-Details.aspx?id=1015719
884754,Yummy Cookies,"[butter, sugar, brown sugar, eggs, vanilla, fl...",www.cookbooks.com/Recipe-Details.aspx?id=727024
2162711,Chocolate Chip Cookies,"[flour, baking soda, salt, margarine, brown su...",www.food.com/recipe/chocolate-chip-cookies-157315
32799,Peanut Butter Cookies,"[sugar, brown sugar, peanut butter, shortening...",www.cookbooks.com/Recipe-Details.aspx?id=667390
229421,Oatmeal Chocolate Chip Cookies,"[shortening softened, brown sugar, granulated ...",www.cookbooks.com/Recipe-Details.aspx?id=278039


Now need to choose what recipes to keep. Possibly one from each cluster. (Most ingredients)

In [ ]:
df["ingredient_count"] = df["ingredients_clean"].apply(len)
df["directions_length"] = df["directions_clean"].apply(len)

clustered_hybrid = recipe_clusters_hybrid.merge(
    df[["title", "ingredient_count", "directions_length"]],
    left_on="recipe_id",
    right_index=True
)

keepers_hybrid = (
    clustered_hybrid
    .sort_values(
        ["cluster_id", "ingredient_count", "directions_length"],
        ascending=[True, False, False]
    )
    .groupby("cluster_id")
    .head(1)["recipe_id"]
)

keepers_hybrid = set(keepers_hybrid)

Mark recipes to keep or remove

In [ ]:
df["hybrid_cluster_id"] = pd.NA
df["is_hybrid_duplicate_candidate"] = False
df["keep_hybrid_representative"] = True

for cluster_id, group in recipe_clusters_hybrid.groupby("cluster_id"):
    ids = group["recipe_id"].tolist()
    keeper = next(recipe_id for recipe_id in ids if recipe_id in keepers_hybrid)

    df.loc[ids, "hybrid_cluster_id"] = cluster_id
    df.loc[ids, "is_hybrid_duplicate_candidate"] = True
    df.loc[ids, "keep_hybrid_representative"] = False
    df.loc[keeper, "keep_hybrid_representative"] = True

Create trimmed dataset

In [ ]:
df_trimmed_hybrid = df[df["keep_hybrid_representative"]].copy()

df_trimmed_hybrid.to_parquet("recipes_trimmed_hybrid_v1.parquet", index=False)

df.shape, df_trimmed_hybrid.shape

inspect dataset

In [ ]:
print(df.shape, df_trimmed_hybrid.shape)
df_trimmed_hybrid.head()

(2231142, 16) (1824097, 16)


,Unnamed: 0,title,ingredients,directions,link,source,NER,title_clean,directions_clean,ingredients_clean,semantic_text,ingredient_count,directions_length,duplicate_cluster_id,is_duplicate_candidate,keep_representative
0,0,No-Bake Nut Cookies,"[1 c. firmly packed brown sugar, 1/2 c. evapor...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu...",no-bake nut cookies,"[""in a heavy 2-quart saucepan, mix brown sugar...","[firmly packed brown sugar, evaporated milk, v...",no-bake nut cookies | bite size shredded rice ...,6,357,<NA>,False,True
1,1,Jewell Ball'S Chicken,"[1 small jar chipped beef, cut up, 4 boned chi...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom...",jewell ball's chicken,"[""place chipped beef on bottom of baking dish....","[small chipped beef cut up, boned chicken brea...",jewell ball's chicken | boned chicken breasts ...,4,175,<NA>,False,True
2,2,Creamy Corn,"[2 (16 oz.) pkg. frozen corn, 1 (8 oz.) pkg. c...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar...",creamy corn,"[""in a slow cooker, combine all ingredients. c...","[frozen corn, cream cheese cubed, butter cubed...",creamy corn | butter cubed cream cheese cubed ...,6,171,0,True,True
3,3,Chicken Funny,"[1 large whole chicken, 2 (10 1/2 oz.) cans ch...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo...",chicken funny,"[""boil and debone chicken."", ""put bite size pi...","[large whole chicken, chicken gravy, cream of ...",chicken funny | box stove top stuffing chicken...,5,394,<NA>,False,True
5,5,Cheeseburger Potato Soup,"[6 baking potatoes, 1 lb. of extra lean ground...","[""Wash potatoes; prick several times with a fo...",www.cookbooks.com/Recipe-Details.aspx?id=20115,Gathered,"[""baking potatoes"", ""extra lean ground beef"", ...",cheeseburger potato soup,"[""wash potatoes; prick several times with a fo...","[baking potatoes, of extra lean ground beef, b...",cheeseburger potato soup | baking potatoes but...,10,904,<NA>,False,True


Change of around -400,000 rows after cleanup.

Load USDA Tables (for portions and nutrients)

In [52]:
food = pd.read_csv("Data/All_Food_Data_April_2026/food.csv")
food_portion = pd.read_csv("Data/All_Food_Data_April_2026/food_portion.csv")
food_nutrient = pd.read_csv("Data/All_Food_Data_April_2026/food_nutrient.csv")

C:\Users\winni\AppData\Local\Temp\ipykernel_51364\3207577813.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  food = pd.read_csv("Data/All_Food_Data_April_2026/food.csv")
C:\Users\winni\AppData\Local\Temp\ipykernel_51364\3207577813.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrient = pd.read_csv("Data/All_Food_Data_April_2026/food_nutrient.csv")


Join food to portions and nutrients

In [53]:
portion_lookup = food_portion.merge(
    food[["fdc_id", "description", "data_type", "food_category_id"]],
    on="fdc_id",
    how="left"
)

nutrient_lookup = food_nutrient.merge(
    food[["fdc_id", "description", "data_type", "food_category_id"]],
    on="fdc_id",
    how="left"
)

Get unique ingredients

In [54]:
unique_ingredients = (
    pd.Series(np.concatenate(df["ingredients_clean"].values))
    .dropna()
    .astype(str)
    .str.strip()
)

unique_ingredients = unique_ingredients[unique_ingredients != ""].drop_duplicates()

unique_ingredients = pd.DataFrame({
    "ingredient_clean": unique_ingredients
})

unique_ingredients.head()

,ingredient_clean
0,firmly packed brown sugar
1,evaporated milk
2,vanilla
3,broken nuts
4,butter or margarine


In [55]:
import re

def simple_clean(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = " ".join(text.split())
    return text

food["description_clean"] = food["description"].apply(simple_clean)

food_search = food[["fdc_id", "description", "description_clean", "data_type", "food_category_id"]].dropna()
food_search.head()

,fdc_id,description,description_clean,data_type,food_category_id
0,1105904,WESSON Vegetable Oil 1 GAL,wesson vegetable oil gal,branded_food,Oils Edible
1,1105905,SWANSON BROTH BEEF,swanson broth beef,branded_food,Herbs/Spices/Extracts
2,1105906,CAMPBELL'S SLOW KETTLE SOUP CLAM CHOWDER,campbell s slow kettle soup clam chowder,branded_food,Prepared Soups
3,1105907,CAMPBELL'S SLOW KETTLE SOUP CHEESE BROCCOLI,campbell s slow kettle soup cheese broccoli,branded_food,Prepared Soups
9,1105908,SWANSON BROTH CHICKEN,swanson broth chicken,branded_food,Herbs/Spices/Extracts


small batch test ingredient matching

In [58]:
food_choices = food_search["description_clean"].tolist()

def match_food(ingredient):
    match = process.extractOne(
        ingredient,
        food_choices,
        scorer=fuzz.WRatio
    )

    if match is None:
        return pd.Series([None, None, None])

    matched_text, score, idx = match
    matched_row = food_search.iloc[idx]

    return pd.Series([
        matched_row["fdc_id"],
        matched_row["description"],
        score
    ])

test_matches = unique_ingredients.head(20).copy()

test_matches[["matched_fdc_id", "matched_description", "match_score"]] = (
    test_matches["ingredient_clean"].apply(match_food)
)

test_matches

,ingredient_clean,matched_fdc_id,matched_description,match_score
0,firmly packed brown sugar,2192147,"BROWN SUGAR, BROWN",95.000000
1,evaporated milk,1116552,EVAPORATED MILK,100.000000
2,vanilla,957385,VANILLA,100.000000
3,broken nuts,424100,NUTS,90.000000
4,butter or margarine,1110048,BUTTER,90.000000
5,bite size shredded rice biscuits,1118719,RICE,90.000000
6,small chipped beef cut up,1110365,BEEF,90.000000
7,boned chicken breasts,463636,CHICKEN BREASTS,95.000000
8,cream of mushroom soup,383865,CREAM OF MUSHROOM SOUP,100.000000
9,carton sour cream,1017201,"SOUR CREAM, SOUR",95.000000


full matching

In [ ]:
#Disregard this cell. Full matching was not yet optimized

ingredient_food_map = unique_ingredients.copy()

ingredient_food_map[["matched_fdc_id", "matched_description", "match_score"]] = (
    ingredient_food_map["ingredient_clean"].apply(match_food)
)

ingredient_food_map.to_parquet("Data/ingredient_food_map_v1.parquet")

ingredient_food_map.head()

KeyboardInterrupt: 

try faiss and embed for full mapping

In [60]:
food_texts = food_search["description_clean"].tolist()

food_embeddings = model.encode(
    food_texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

food_embeddings = food_embeddings.astype("float32")

np.save("Data/food_description_embeddings.npy", food_embeddings)

Batches: 100%|██████████| 8152/8152 [29:40<00:00,  4.58it/s]  


In [61]:
food_index = faiss.IndexFlatIP(food_embeddings.shape[1])
food_index.add(food_embeddings)

faiss.write_index(food_index, "Data/food_description_index.faiss")

In [62]:
ingredient_texts = unique_ingredients["ingredient_clean"].tolist()

ingredient_embeddings = model.encode(
    ingredient_texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

ingredient_embeddings = ingredient_embeddings.astype("float32")

np.save("Data/ingredient_embeddings.npy", ingredient_embeddings)

Batches: 100%|██████████| 7602/7602 [24:28<00:00,  5.18it/s]   


In [63]:
scores, idxs = food_index.search(ingredient_embeddings, k=5)

In [69]:
np.save("Data/ingredient_food_scores.npy", scores)
np.save("Data/ingredient_food_idxs.npy", idxs)

to load later

In [70]:
scores = np.load("Data/ingredient_food_scores.npy")
idxs = np.load("Data/ingredient_food_idxs.npy")

Create mapping

In [ ]:
rows = []

for ingredient_i, ingredient in enumerate(ingredient_texts):
    for rank in range(5):
        food_i = idxs[ingredient_i, rank]

        rows.append({
            "ingredient_clean": ingredient,
            "rank": rank + 1,
            "matched_fdc_id": food_search.iloc[food_i]["fdc_id"],
            "matched_description": food_search.iloc[food_i]["description"],
            "matched_description_clean": food_search.iloc[food_i]["description_clean"],
            "data_type": food_search.iloc[food_i]["data_type"],
            "food_category_id": food_search.iloc[food_i]["food_category_id"],
            "similarity": float(scores[ingredient_i, rank])
        })

ingredient_food_candidates = pd.DataFrame(rows)

#ingredient_food_candidates.to_parquet("Data/ingredient_food_candidates_v1.parquet")

#ingredient_food_candidates.head(20)

ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column food_category_id with type object')

In [66]:
ingredient_food_candidates["food_category_id"] = (
    ingredient_food_candidates["food_category_id"]
    .astype("string")
)

ingredient_food_candidates["matched_fdc_id"] = pd.to_numeric(
    ingredient_food_candidates["matched_fdc_id"],
    errors="coerce"
).astype("Int64")

ingredient_food_candidates["rank"] = pd.to_numeric(
    ingredient_food_candidates["rank"],
    errors="coerce"
).astype("Int64")

ingredient_food_candidates["similarity"] = pd.to_numeric(
    ingredient_food_candidates["similarity"],
    errors="coerce"
).astype("float32")

for col in [
    "ingredient_clean",
    "matched_description",
    "matched_description_clean",
    "data_type"
]:
    ingredient_food_candidates[col] = (
        ingredient_food_candidates[col]
        .astype("string")
    )

In [67]:
ingredient_food_candidates.to_parquet(
    "Data/ingredient_food_candidates_v1.parquet",
    index=False
)

review

In [68]:
ingredient_food_candidates[
    ingredient_food_candidates["rank"] == 1
].sort_values("similarity").head(50)

,ingredient_clean,rank,matched_fdc_id,matched_description,matched_description_clean,data_type,food_category_id,similarity
4144680,internet marketing and search engine optimizat...,1,1467855,"TRADER'S CHOICE, PAPRIKA",trader s choice paprika,branded_food,Herbs & Spices,0.252089
4191915,ratings not yet rated read comments,1,1882732,GRADE A,grade a,branded_food,Milk,0.255228
3952690,us cellular wireless towers out,1,737334,TIME OUTS,time outs,branded_food,Frozen Appetizers & Hors D'oeuvres,0.256714
1996940,understanding for medicare billers secretaries...,1,2081590,KAISER PREMIUM BUNS,kaiser premium buns,branded_food,Breads & Buns,0.270368
4077330,see applications for more details,1,405408,VIA INSTANT REFRESHERS,via instant refreshers,branded_food,Coffee,0.271188
4072935,we are fast approaching our first full year of...,1,2184300,V8 SPL TROP BLD 5% JCE 96FO,v spl trop bld jce fo,branded_food,Non Alcoholic Beverages Ready to Drink,0.276153
2038595,psalms use either or both,1,2272241,"DOUBLE CHOCOLATE BETTER COOKIE, DOUBLE CHOCOLATE",double chocolate better cookie double chocolate,branded_food,Cookies & Biscuits,0.277984
4176830,domain registration,1,1947606,"VANILLA CAKE WITH BUTTERCREAM ICING, VANILLA",vanilla cake with buttercream icing vanilla,branded_food,"Cakes, Cupcakes, Snack Cakes",0.279122
4346940,comments leave comment,1,2706211,Spam,spam,survey_fndds_food,2602,0.280044
2689695,you will need,1,430130,GOOD LUCK CANDY,good luck candy,branded_food,Candy,0.280164


In [12]:
ingredient_food_candidates = pd.read_parquet(
    "Data/ingredient_food_candidates_v1.parquet"
)

best_ingredient_matches = (
    ingredient_food_candidates
    .sort_values(["ingredient_clean", "rank"])
    .groupby("ingredient_clean")
    .head(1)
    .copy()
)

best_ingredient_matches.head()

,ingredient_clean,rank,matched_fdc_id,matched_description,matched_description_clean,data_type,food_category_id,similarity
156020,a,1,2672037,A,a,branded_food,"Fruit & Vegetable Juice, Nectars & Fruit Drinks",1.000000
4131350,a a almond milk,1,1568995,ALMOND MILK,almond milk,branded_food,Plant Based Milk,0.913217
4248715,a a apple cider vinegar,1,1950010,"VINEGAR, APPLE CIDER",vinegar apple cider,branded_food,Other Cooking Sauces,0.884126
4206185,a a arborio or italian style rice,1,2570047,ARBORIO ITALIAN-STYLE RICE,arborio italian style rice,branded_food,Rice,0.949407
4628080,a a balsamic vinegar,1,1554628,THE BALSAMIC VINEGAR,the balsamic vinegar,branded_food,Other Condiments,0.941221


inspect match quality

In [13]:
best_ingredient_matches["similarity"].describe()

count    1.945862e+06
mean     7.862180e-01
std      9.499627e-02
min      2.520887e-01
25%      7.269646e-01
50%      7.915011e-01
75%      8.525380e-01
max      1.000001e+00
Name: similarity, dtype: float64

weaker matches

In [14]:
best_ingredient_matches.sort_values("similarity").head(50)

,ingredient_clean,rank,matched_fdc_id,matched_description,matched_description_clean,data_type,food_category_id,similarity
4144680,internet marketing and search engine optimizat...,1,1467855,"TRADER'S CHOICE, PAPRIKA",trader s choice paprika,branded_food,Herbs & Spices,0.252089
4191915,ratings not yet rated read comments,1,1882732,GRADE A,grade a,branded_food,Milk,0.255228
3952690,us cellular wireless towers out,1,737334,TIME OUTS,time outs,branded_food,Frozen Appetizers & Hors D'oeuvres,0.256714
1996940,understanding for medicare billers secretaries...,1,2081590,KAISER PREMIUM BUNS,kaiser premium buns,branded_food,Breads & Buns,0.270368
4077330,see applications for more details,1,405408,VIA INSTANT REFRESHERS,via instant refreshers,branded_food,Coffee,0.271188
4072935,we are fast approaching our first full year of...,1,2184300,V8 SPL TROP BLD 5% JCE 96FO,v spl trop bld jce fo,branded_food,Non Alcoholic Beverages Ready to Drink,0.276153
2038595,psalms use either or both,1,2272241,"DOUBLE CHOCOLATE BETTER COOKIE, DOUBLE CHOCOLATE",double chocolate better cookie double chocolate,branded_food,Cookies & Biscuits,0.277984
4176830,domain registration,1,1947606,"VANILLA CAKE WITH BUTTERCREAM ICING, VANILLA",vanilla cake with buttercream icing vanilla,branded_food,"Cakes, Cupcakes, Snack Cakes",0.279122
4346940,comments leave comment,1,2706211,Spam,spam,survey_fndds_food,2602,0.280044
2689695,you will need,1,430130,GOOD LUCK CANDY,good luck candy,branded_food,Candy,0.280164


best matches

In [15]:
best_ingredient_matches.sort_values(
    "similarity",
    ascending=False
).head(50)

,ingredient_clean,rank,matched_fdc_id,matched_description,matched_description_clean,data_type,food_category_id,similarity
4924810,yellow beets,1,2440563,YELLOW BEETS,yellow beets,branded_food,"Seasoning Mixes, Salts, Marinades & Tenderizers",1.000001
705895,tonic water,1,424055,TONIC WATER,tonic water,branded_food,Water,1.000001
5335325,lemon lime seltzer water,1,1518355,LEMON LIME SELTZER WATER,lemon lime seltzer water,branded_food,Water,1.000001
91660,freeze dried chives,1,1668464,FREEZE DRIED CHIVES,freeze dried chives,branded_food,Pre-Packaged Fruit & Vegetables,1.000001
71620,diced water chestnuts,1,1972162,DICED WATER CHESTNUTS,diced water chestnuts,branded_food,Canned Vegetables,1.000001
6405355,salsa sweet potato,1,1896310,"SALSA, SWEET POTATO",salsa sweet potato,branded_food,Dips & Salsa,1.000001
106230,caesar croutons,1,505711,CAESAR CROUTONS,caesar croutons,branded_food,Breads & Buns,1.000001
4056875,sweet red pepper sauce,1,2023879,SWEET RED PEPPER SAUCE,sweet red pepper sauce,branded_food,"Ketchup, Mustard, BBQ & Cheese Sauce",1.000001
2698175,brioche buns,1,521928,BRIOCHE BUNS,brioche buns,branded_food,Breads & Buns,1.000001
1257050,cranberry chutney,1,958079,CRANBERRY CHUTNEY,cranberry chutney,branded_food,"Pickles, Olives, Peppers & Relishes",1.000001


middle values

In [16]:
best_ingredient_matches[
    (best_ingredient_matches["similarity"] > 0.60) &
    (best_ingredient_matches["similarity"] < 0.80)
].sample(50)

,ingredient_clean,rank,matched_fdc_id,matched_description,matched_description_clean,data_type,food_category_id,similarity
6768555,avocado thinly sliced whole foods ea for thru,1,2420555,CHUNKED AVOCADO,chunked avocado,branded_food,Frozen Fruit & Fruit Juice Concentrates,0.751981
6512060,filet mignon or possibly sirloin cut about in ...,1,375115,VEGETABLE SKILLET SAUCE,vegetable skillet sauce,branded_food,Other Cooking Sauces,0.692457
74620,chicken or beef,1,410520,BEEF,beef,branded_food,Canned Meat,0.785974
6148625,onion fried paste,1,2028283,"FRIED CHILI PASTE SALSA, FRIED CHILI PASTE",fried chili paste salsa fried chili paste,branded_food,"Oriental, Mexican & Ethnic Sauces",0.777711
6457365,feta about cubed,1,1898250,FETA CHEESE CUBES,feta cheese cubes,branded_food,Cheese,0.731421
6033525,bunch italian kale or escarole chopped,1,591172,CHOPPED KALE,chopped kale,branded_food,Frozen Vegetables,0.766061
1795635,wondra flour to taste,1,2016725,"FOOD LION, ALL-PURPOSE FLOUR",food lion all purpose flour,branded_food,Flours & Corn Meal,0.620607
2320710,granny smith apple peeled cored and cut into c...,1,1268988,SLICED GRANNY SMITH APPLES,sliced granny smith apples,branded_food,Pre-Packaged Fruit & Vegetables,0.770153
438515,full grocery bag of popped corn,1,546329,HALF-POPPED CORN,half popped corn,branded_food,"Popcorn, Peanuts, Seeds & Related Snacks",0.766869
8216605,refrigerated pie dough sheets,1,2231027,Pillsbury Premade Refrigerated Pie Crusts,pillsbury premade refrigerated pie crusts,branded_food,Baking/Cooking Mixes/Supplies,0.763879


accepted match table

In [17]:
accepted_ingredient_matches = best_ingredient_matches[
    best_ingredient_matches["similarity"] >= 0.72
].copy()

accepted_ingredient_matches.to_parquet(
    "Data/accepted_ingredient_matches_v1.parquet",
    index=False
)

accepted_ingredient_matches.shape

(1501620, 8)